# 02 Prep: features, links, folds and negatives

Builds everything the models need and saves it to `prep/`:
- integer index for every herb, compound and target
- compound features computed from SMILES with RDKit, target features from AAC and ESM-2, herb features from plant family
- H-C and C-T links
- 5-fold cross validation for four tasks, with negatives drawn from the unlabeled pool

Reads: the dataset files in the project folder
Writes: `prep/meta.json`, `prep/features.npz`, `prep/links.npz`, `prep/folds_<task>.npz`

## Setup

In [1]:
import hti
import json
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from hti.data import DATA, PREP, TASKS

RDLogger.DisableLog('rdApp.*')

SEED = 42
K = 5                  # folds; every test link (or compound / target) is tested exactly once
VAL_SHARE = 0.15       # share of all links that goes to validation in each fold
NEG_PER_POS = {'train': 10, 'val': 10, 'test': 50}

PREP.mkdir(exist_ok=True)
for old in PREP.glob('*'):
    if old.is_file():
        old.unlink()       # files from earlier versions of the pipeline

## Load the dataset files

In [2]:
herbs = pd.read_csv(DATA / 'herbs.tsv', sep='\t', dtype=str)
compounds = pd.read_csv(DATA / 'compounds.tsv', sep='\t', dtype=str)
hc_rows = pd.read_csv(DATA / 'herb_compound_edges.tsv', sep='\t', dtype=str, usecols=['herb_id', 'compound_id'])
ct_rows = pd.read_csv(DATA / 'compound_target_labeled.csv', dtype=str, usecols=['compound_id', 'protein_id'])
aac = pd.read_csv(DATA / 'target_features_aac.csv')
esm = pd.read_csv(DATA / 'target_features_esm2.csv')
print(len(herbs), 'herbs,', len(compounds), 'compounds,', len(aac), 'targets')

62 herbs, 2319 compounds, 1341 targets


## Integer index for every node
Herbs, compounds and targets are each sorted by id; the position in that order is the node index.

In [3]:
herb_ids = sorted(herbs['herb_id'])
compound_ids = sorted(compounds['compound_id'])
target_ids = sorted(aac['protein_id'])
herb_index = {x: i for i, x in enumerate(herb_ids)}
compound_index = {x: i for i, x in enumerate(compound_ids)}
target_index = {x: i for i, x in enumerate(target_ids)}
n_h, n_c, n_t = len(herb_ids), len(compound_ids), len(target_ids)
print(n_h, n_c, n_t)

62 2319 1341


## Compound features from SMILES
- ECFP4 bits (2,048): input of the graph model and of the similarity kNN
- ECFP4 counts (log scale), FCFP4 bits and MACCS keys (4,263 together): input of the two-tower model

The ECFP4 bits are compared with the rows of `compound_features_ecfp4.csv` to confirm they are the same fingerprints.

In [4]:
smiles = compounds.set_index('compound_id')['Canonical_SMILES']
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
feature_morgan = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, atomInvariantsGenerator=rdFingerprintGenerator.GetMorganFeatureAtomInvGen())

ecfp4, ecfp4_counts, fcfp4, maccs = [], [], [], []
for cid in compound_ids:
    mol = Chem.MolFromSmiles(smiles[cid])
    ecfp4.append(morgan.GetFingerprintAsNumPy(mol))
    ecfp4_counts.append(np.log1p(morgan.GetCountFingerprintAsNumPy(mol)))
    fcfp4.append(feature_morgan.GetFingerprintAsNumPy(mol))
    maccs.append(np.array(list(MACCSkeys.GenMACCSKeys(mol)), dtype=np.uint8))

ecfp4 = np.array(ecfp4, dtype=np.uint8)
chem = np.hstack([np.array(ecfp4_counts), np.array(fcfp4), np.array(maccs)]).astype(np.float32)

file_bits = pd.read_csv(DATA / 'compound_features_ecfp4.csv').drop(columns='compound_id').to_numpy().astype(np.uint8)
assert np.array_equal(ecfp4, file_bits), 'recomputed ECFP4 does not match the fingerprint file rows'
print('ECFP4', ecfp4.shape, ' two-tower chemistry', chem.shape)

ECFP4 (2319, 2048)  two-tower chemistry (2319, 4263)


## Target and herb features

In [5]:
aac_x = aac.set_index('protein_id').loc[target_ids].filter(like='aac_').to_numpy(np.float32)
esm_x = esm.set_index('protein_id').loc[target_ids].filter(like='esm_').to_numpy(np.float32)

family_names = sorted(herbs['Family'].unique())
family_of = herbs.set_index('herb_id')['Family']
family = np.zeros((n_h, len(family_names)), dtype=np.float32)
for h in herb_ids:
    family[herb_index[h], family_names.index(family_of[h])] = 1
print('AAC', aac_x.shape, ' ESM-2', esm_x.shape, ' family one-hot', family.shape)

AAC (1341, 20)  ESM-2 (1341, 1280)  family one-hot (62, 38)


## Links
The H-C file lists a pair once per plant part, so pairs are de-duplicated.

In [6]:
hc_unique = hc_rows.drop_duplicates()
hc = np.stack([hc_unique['herb_id'].map(herb_index), hc_unique['compound_id'].map(compound_index)]).astype(np.int64)
ct = np.stack([ct_rows['compound_id'].map(compound_index), ct_rows['protein_id'].map(target_index)]).astype(np.int64)
assert hc.min() >= 0 and ct.min() >= 0

herb_member = np.zeros((n_c, n_h), dtype=np.float32)     # which herbs each compound is found in
herb_member[hc[1], hc[0]] = 1

known_ct = np.zeros((n_c, n_t), dtype=bool)
known_ct[ct[0], ct[1]] = True
known_hc = np.zeros((n_h, n_c), dtype=bool)
known_hc[hc[0], hc[1]] = True
print('H-C links', hc.shape[1], ' C-T links', ct.shape[1])

H-C links 5005  C-T links 14166


## Negative pools
- C-T: the 3.1 million pairs of `compound_target_unlabeled.csv`, checked to be every pair that is not a known link
- H-C: there is no unlabeled file, so the pool is every herb x compound pair without a link

In [7]:
unlabeled = pd.read_csv(DATA / 'compound_target_unlabeled.csv', usecols=['compound_id', 'protein_id'], dtype='category')

def codes(col, index):
    lookup = np.array([index[x] for x in col.cat.categories])
    return lookup[col.cat.codes.to_numpy()]

ct_pool = np.sort(codes(unlabeled['compound_id'], compound_index) * n_t + codes(unlabeled['protein_id'], target_index))
del unlabeled
assert np.array_equal(ct_pool, np.flatnonzero(~known_ct.ravel())), 'unlabeled file is not the complement of the known links'
hc_pool = np.flatnonzero(~known_hc.ravel())

pool_compound, pool_target = ct_pool // n_t, ct_pool % n_t
labeled_compound = np.zeros(n_c, dtype=bool)
labeled_compound[ct[0]] = True
print('C-T pool', len(ct_pool), ' H-C pool', len(hc_pool))

C-T pool 3095613  H-C pool 138773


## Compound groups for the new-compound split
Compounds with identical ECFP4 bits, or with the same InChIKey first block, are joined into one group.
A whole group always lands in the same part (train, val or test), so a test compound never has a twin in training.

In [8]:
fp_key = pd.factorize(pd.Series([r.tobytes() for r in ecfp4]))[0]
inchi_key = pd.factorize(compounds.set_index('compound_id').loc[compound_ids, 'InChIKey'].str[:14])[0]

src, dst = [], []
for key in (fp_key, inchi_key):
    order = np.argsort(key, kind='stable')
    same = key[order][1:] == key[order][:-1]
    src.append(order[1:][same])
    dst.append(order[:-1][same])
src, dst = np.concatenate(src), np.concatenate(dst)
_, compound_group = connected_components(coo_matrix((np.ones(len(src)), (src, dst)), shape=(n_c, n_c)), directed=False)
sizes = np.bincount(compound_group)
print('compounds in a group of 2 or more:', int((sizes[compound_group] > 1).sum()))

compounds in a group of 2 or more: 471


## Fold assignment
Units (links, compounds or targets) are grouped, and groups are spread over 5 parts so every part holds about 20% of the links.
For fold k, part k is the test set. From the other four parts, groups are drawn at random until 15% of all links are in validation.
Everything left is training. So each fold is about 65% train, 15% validation and 20% test, and every unit is tested exactly once.

In [9]:
def make_folds(groups, weights, rng):
    """groups: group id (0..G-1) of every unit, weights: links per unit. Returns (train, val, test) unit masks per fold."""
    n_groups = groups.max() + 1
    group_weight = np.bincount(groups, weights=weights, minlength=n_groups)
    order = np.lexsort((rng.random(n_groups), -group_weight))    # biggest groups first, random among equals
    part_of_group, load = np.empty(n_groups, dtype=int), np.zeros(K)
    for g in order:
        p = int(np.argmin(load))
        part_of_group[g] = p
        load[p] += group_weight[g]
    folds = []
    for k in range(K):
        rest = rng.permutation(np.flatnonzero(part_of_group != k))
        taken = np.cumsum(group_weight[rest])
        val_groups = rest[: int(np.searchsorted(taken, VAL_SHARE * group_weight.sum())) + 1]
        is_test = part_of_group[groups] == k
        is_val = np.isin(groups, val_groups)
        folds.append((~is_test & ~is_val, is_val, is_test))
    return folds


def link_folds(task, rng):
    """(train, val, test) masks over the task's links, plus the units of each part"""
    if task in ('ct_edge', 'hc_edge'):
        n = ct.shape[1] if task == 'ct_edge' else hc.shape[1]
        return [(m, None) for m in make_folds(np.arange(n), np.ones(n), rng)]
    axis, n_units = (0, n_c) if task == 'ct_compound' else (1, n_t)
    units = np.unique(ct[axis])
    groups = pd.factorize(compound_group[units])[0] if task == 'ct_compound' else np.arange(len(units))
    weights = np.bincount(ct[axis], minlength=n_units)[units]
    out = []
    for unit_masks in make_folds(groups, weights, rng):
        part_of_unit = np.full(n_units, -1)
        for s, m in enumerate(unit_masks):
            part_of_unit[units[m]] = s
        link_part = part_of_unit[ct[axis]]
        out.append((tuple(link_part == s for s in range(3)), [units[m] for m in unit_masks]))
    return out

## Negatives
For each fold and each part (train, val, test):
- **neg**: negatives for PR-AUC, 10 per positive for train and val and 50 per positive for test, drawn at random without replacement.
  The three parts never share a negative.
- **pair**: one negative column for every positive, for the same row (same compound, or same herb for H-C). It gives the paired BPR loss,
  so loss can be reported for train, val and test in the same way.

Which pairs a part may draw from:
- C-T hidden links: unlabeled pairs of the 726 compounds that have known targets. Unlabeled pairs of compounds with no known target at all
  would be too easy to rule out and would inflate PR-AUC.
- C-T new compounds: unlabeled pairs of that part's compounds.
- C-T new targets: unlabeled pairs of that part's targets (with compounds that have known targets). Ranking is also only among that part's targets.
- H-C hidden links: herb x compound pairs without a link.

In [10]:
def draw(pool, n, rng, note):
    if n > len(pool):
        print(f'   {note}: pool has {len(pool)} pairs, {n} wanted, all of them are used')
        n = len(pool)
    return rng.choice(pool, size=n, replace=False)


def paired_negatives(pos, known, allowed, rng):
    rows = pos[0]
    cand = rng.integers(0, known.shape[1], len(rows))
    for _ in range(200):
        bad = known[rows, cand]
        if allowed is not None:
            bad |= ~allowed[cand]
        if not bad.any():
            break
        cand[bad] = rng.integers(0, known.shape[1], int(bad.sum()))
    assert not bad.any()
    return cand


def to_pairs(keys, n_cols):
    return np.stack([keys // n_cols, keys % n_cols])

## Build and save the folds of all four tasks

In [11]:
summary = []
for task in TASKS:
    rng = np.random.default_rng(SEED + TASKS.index(task))
    links, known, n_cols = (hc, known_hc, n_c) if task == 'hc_edge' else (ct, known_ct, n_t)
    store = {}
    for k, (masks, units) in enumerate(link_folds(task, rng)):
        pos = {s: links[:, m] for s, m in zip(['train', 'val', 'test'], masks)}
        need = {s: NEG_PER_POS[s] * pos[s].shape[1] for s in pos}
        allowed = {s: None for s in pos}

        if task in ('ct_edge', 'hc_edge'):
            pool = ct_pool[labeled_compound[pool_compound]] if task == 'ct_edge' else hc_pool
            picked = draw(pool, sum(need.values()), rng, f'{task} fold {k}')
            cut = np.cumsum([need['train'], need['val']])
            neg = dict(zip(['train', 'val', 'test'], np.split(picked, cut)))
        else:
            neg = {}
            for s, u in zip(['train', 'val', 'test'], units):
                if task == 'ct_compound':
                    pool = ct_pool[np.isin(pool_compound, u)]
                else:
                    pool = ct_pool[np.isin(pool_target, u) & labeled_compound[pool_compound]]
                    allowed[s] = np.isin(np.arange(n_t), u)
                neg[s] = draw(pool, need[s], rng, f'{task} fold {k} {s}')

        for s in ['train', 'val', 'test']:
            store[f'f{k}_{s}_pos'] = pos[s].astype(np.int32)
            store[f'f{k}_{s}_neg'] = to_pairs(neg[s], n_cols).astype(np.int32)
            store[f'f{k}_{s}_pair'] = paired_negatives(pos[s], known, allowed[s], rng).astype(np.int32)
            store[f'f{k}_{s}_cols'] = allowed[s] if allowed[s] is not None else np.zeros(0, dtype=bool)
            summary.append({'task': task, 'fold': k, 'part': s, 'positives': pos[s].shape[1],
                            'negatives': len(neg[s]), 'neg per pos': round(len(neg[s]) / pos[s].shape[1], 1)})
    np.savez_compressed(PREP / f'folds_{task}.npz', **store)
    print('saved', task)

summary = pd.DataFrame(summary)
summary.pivot_table(index=['task', 'fold'], columns='part', values='positives')

saved ct_edge
saved ct_compound
saved ct_target
saved hc_edge


part                test   train     val
task        fold                        
ct_compound 0     2834.0  9168.0  2164.0
            1     2833.0  9187.0  2146.0
            2     2833.0  9205.0  2128.0
            3     2833.0  9163.0  2170.0
            4     2833.0  9164.0  2169.0
ct_edge     0     2834.0  9207.0  2125.0
            1     2833.0  9208.0  2125.0
            2     2833.0  9208.0  2125.0
            3     2833.0  9208.0  2125.0
            4     2833.0  9208.0  2125.0
ct_target   0     2834.0  9174.0  2158.0
            1     2833.0  9172.0  2161.0
            2     2833.0  9206.0  2127.0
            3     2833.0  9203.0  2130.0
            4     2833.0  9096.0  2237.0
hc_edge     0     1001.0  3253.0   751.0
            1     1001.0  3253.0   751.0
            2     1001.0  3253.0   751.0
            3     1001.0  3253.0   751.0
            4     1001.0  3253.0   751.0

## Save features, links and the index

In [12]:
np.savez_compressed(PREP / 'features.npz', ecfp4=ecfp4, chem=chem, aac=aac_x, esm=esm_x,
                    family=family, herb_member=herb_member)
np.savez_compressed(PREP / 'links.npz', hc=hc, ct=ct)
(PREP / 'meta.json').write_text(json.dumps({
    'seed': SEED, 'k': K, 'val_share': VAL_SHARE, 'neg_per_pos': NEG_PER_POS,
    'n_herbs': n_h, 'n_compounds': n_c, 'n_targets': n_t,
    'herb_ids': herb_ids, 'compound_ids': compound_ids, 'target_ids': target_ids, 'families': family_names,
}, indent=1))
print(sorted(p.name for p in PREP.iterdir()))

['features.npz', 'folds_ct_compound.npz', 'folds_ct_edge.npz', 'folds_ct_target.npz', 'folds_hc_edge.npz', 'links.npz', 'meta.json']


## Leakage and consistency checks on the saved folds
Every check must be True.

In [13]:
def keys(pairs, n_cols):
    return pairs[0].astype(np.int64) * n_cols + pairs[1]

rows = []
for task in TASKS:
    z = np.load(PREP / f'folds_{task}.npz')
    links, known, n_cols = (hc, known_hc, n_c) if task == 'hc_edge' else (ct, known_ct, n_t)
    all_keys = keys(links, n_cols)
    tested = []
    for k in range(K):
        p = {s: keys(z[f'f{k}_{s}_pos'], n_cols) for s in ['train', 'val', 'test']}
        n = {s: keys(z[f'f{k}_{s}_neg'], n_cols) for s in ['train', 'val', 'test']}
        pr = {s: z[f'f{k}_{s}_pair'] for s in ['train', 'val', 'test']}
        tested.append(p['test'])
        check = {
            'parts cover all links once': np.array_equal(np.sort(np.concatenate(list(p.values()))), np.sort(all_keys)),
            'no negative is a known link': not any(known.ravel()[v].any() for v in n.values()),
            'parts share no negative': len(np.unique(np.concatenate(list(n.values())))) == sum(len(v) for v in n.values()),
            'paired negatives are unlabeled': all(not known[z[f'f{k}_{s}_pos'][0], pr[s]].any() for s in pr),
        }
        if task == 'ct_compound':
            comp = {s: set(z[f'f{k}_{s}_pos'][0]) for s in p}
            grp = {s: set(compound_group[list(v)]) for s, v in comp.items()}
            check['no compound in two parts'] = not (comp['train'] & comp['val'] or comp['train'] & comp['test'] or comp['val'] & comp['test'])
            check['no compound group split'] = not (grp['train'] & grp['val'] or grp['train'] & grp['test'] or grp['val'] & grp['test'])
        if task == 'ct_target':
            tg = {s: set(z[f'f{k}_{s}_pos'][1]) for s in p}
            check['no target in two parts'] = not (tg['train'] & tg['val'] or tg['train'] & tg['test'] or tg['val'] & tg['test'])
        rows.append({'task': task, 'fold': k, **check})
    tested = np.concatenate(tested)
    rows.append({'task': task, 'fold': 'all', 'every link tested exactly once': len(tested) == len(all_keys) and len(np.unique(tested)) == len(all_keys)})
checks = pd.DataFrame(rows).set_index(['task', 'fold'])
assert (checks.isna() | checks.eq(True)).all().all(), 'a check failed'   # empty cells are checks that do not apply to that task
checks.astype(object).where(checks.notna(), '')

parts cover all links once no negative is a known link  \
task        fold                                                          
ct_edge     0                          True                        True   
            1                          True                        True   
            2                          True                        True   
            3                          True                        True   
            4                          True                        True   
            all                                                           
ct_compound 0                          True                        True   
            1                          True                        True   
            2                          True                        True   
            3                          True                        True   
            4                          True                        True   
            all                                                           
ct_target   0                          True                        True   
            1                          True                        True   
            2                          True                        True   
            3                          True                        True   
            4                          True                        True   
            all                                                           
hc_edge     0                          True                        True   
            1                          True                        True   
            2                          True                        True   
            3                          True                        True   
            4                          True                        True   
            all                                                           

                 parts share no negative paired negatives are unlabeled  \
task        fold                                                          
ct_edge     0                       True                           True   
            1                       True                           True   
            2                       True                           True   
            3                       True                           True   
            4                       True                           True   
            all                                                           
ct_compound 0                       True                           True   
            1                       True                           True   
            2                       True                           True   
            3                       True                           True   
            4                       True                           True   
            all                                                           
ct_target   0                       True                           True   
            1                       True                           True   
            2                       True                           True   
            3                       True                           True   
            4                       True                           True   
            all                                                           
hc_edge     0                       True                           True   
            1                       True                           True   
            2                       True                           True   
            3                       True                           True   
            4                       True                           True   
            all                                                           

                 every link tested exactly once no compound in two parts  \
task        fold                       

## Herb-target test pairs per fold
For the two C-T tasks with herbs, a herb-target pair belongs to a part only if it is reachable through that part's links and not through
an earlier part (train first, then val). So a test herb-target pair has no path through training or validation links.

In [14]:
herb_has = np.zeros((n_h, n_c), dtype=np.float32)
herb_has[hc[0], hc[1]] = 1

def reach(pairs):
    m = np.zeros((n_c, n_t), dtype=np.float32)
    m[pairs[0], pairs[1]] = 1
    return herb_has @ m > 0

rows = []
for task in ['ct_edge', 'ct_compound']:
    z = np.load(PREP / f'folds_{task}.npz')
    for k in range(K):
        tr = reach(z[f'f{k}_train_pos'])
        va = reach(z[f'f{k}_val_pos']) & ~tr
        te = reach(z[f'f{k}_test_pos']) & ~tr & ~va
        rows.append({'task': task, 'fold': k, 'train': int(tr.sum()), 'val': int(va.sum()), 'test': int(te.sum())})
print('all herb-target pairs:', int(reach(ct).sum()))
pd.DataFrame(rows)

all herb-target pairs: 24835


,task,fold,train,val,test
0,ct_edge,0,20060,2114,2661
1,ct_edge,1,20023,2364,2448
2,ct_edge,2,20107,2370,2358
3,ct_edge,3,19807,2565,2463
4,ct_edge,4,19881,2117,2837
5,ct_compound,0,18768,1963,4104
6,ct_compound,1,21492,1255,2088
7,ct_compound,2,21670,1103,2062
8,ct_compound,3,20960,1295,2580
9,ct_compound,4,21721,1442,1672
